# Adaptive Feedback Loop Based Cognitive SOC: Threat Classification
This Jupyter Notebook implements the machine learning threat classification module for the Cognitive SOC.

## Methodology Overview:
1. **Data Sources:**
    - **TIP:** Real-world Indicators of Compromise (IOC) from `Data/IOC.json` (N = 4,200)
2. **Feature Engineering:** Extracts **16 structural/contextual features** (3 indicator type one-hot, 5 threat category one-hot, and 8 scalar features).
    - All TIP-scored fields (`exposure_score`, `confidence`, `security_vendors`) are **excluded** to prevent data leakage.
    - Features are derived solely from raw, structurally observable IOC attributes.
3. **Model:** Random Forest Classifier (`n_estimators=200, max_depth=12, class_weight='balanced'`).
4. **Evaluation:** Classification reports, Confusion Matrix, Feature Importance, and validation checks.

In [ ]:
import os
import json
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set base paths
BASE_DIR = os.getcwd()
IOC_JSON_PATH   = os.path.join(BASE_DIR, "Data", "IOC.json")
MODEL_OUT_PATH  = os.path.join(BASE_DIR, "Data", "ioc_rf_model.pkl")
ENCODER_OUT_PATH= os.path.join(BASE_DIR, "Data", "label_encoder.pkl")

print(f"Workspace directory: {BASE_DIR}")

In [ ]:
# 1. Load & Preprocess TIP real dataset (`IOC.json`)
# NOTE: Only raw/structural fields are used as features.
# TIP-scored fields (exposure_score, confidence, security_vendors)
# are reserved for label derivation only to prevent data leakage.

with open(IOC_JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} raw indicators from IOC.json.")

ext_def = "extension-definition--8e1c4b7d-9a2f-4d63-b5e0-3c7a1f9d2b44"
records = []

for item in raw_data:
    name         = item.get("name", "")
    pattern      = item.get("pattern", "")
    description  = item.get("description", "")
    created_str  = item.get("created",  "2026-06-05T00:08:20.181Z")
    modified_str = item.get("modified", "2026-06-05T00:08:20.181Z")

    props     = item.get("extensions", {}).get(ext_def, {}).get("properties", {})
    score     = props.get("exposure_score", 5)   # label only
    verdict   = props.get("verdict", "Suspicious") # label only
    country   = props.get("country_code", "US")  # raw geo feature
    asn_owner = props.get("asn_owner", "")        # raw network feature

    # ── Indicator type (from STIX pattern structure) ──────────
    ioc_type = "Hash"
    if "ipv4-addr" in pattern or "ipv6-addr" in pattern:
        ioc_type = "IP"
    elif "domain-name" in pattern:
        ioc_type = "Domain"

    # ── Threat category (NLP on description text) ─────────────
    desc_lower = description.lower()
    threat_cat = "Other"
    if "c2" in desc_lower or "command and control" in desc_lower:
        threat_cat = "C2"
    elif "phishing" in desc_lower:
        threat_cat = "Phishing"
    elif "exploit" in desc_lower or "vulnerability" in desc_lower:
        threat_cat = "Exploit"
    elif "malware" in desc_lower or "trojan" in desc_lower or "botnet" in desc_lower:
        threat_cat = "Malware"

    # ── Temporal features (from STIX timestamps) ─────────────
    try:
        dt_c = datetime.strptime(created_str.split(".")[0] + "Z",  "%Y-%m-%dT%H:%M:%SZ")
        dt_m = datetime.strptime(modified_str.split(".")[0] + "Z", "%Y-%m-%dT%H:%M:%SZ")
        days = max(1, (dt_m - dt_c).days)
        hour_created = dt_c.hour / 23.0
    except:
        days = 1
        hour_created = 0.5
    active_days = np.log1p(days)
    first_seen  = active_days * 0.8

    # ── Geolocation risk (raw country_code) ───────────────────
    geo_risk = 0.5
    if country in ["RU", "CN", "KP", "IR", "BY"]:
        geo_risk = 1.0
    elif country in ["US", "CA", "GB", "DE", "FR", "JP", "AU"]:
        geo_risk = 0.0

    # ── ASN reputation (raw asn_owner string) ─────────────────
    asn_rep = 0.5
    if "google" in asn_owner.lower() or "amazon" in asn_owner.lower() or "cloudflare" in asn_owner.lower():
        asn_rep = 0.1
    elif not asn_owner:
        asn_rep = 0.9

    # ── IOC name structural features ──────────────────────────
    name_len     = np.log1p(len(name)) / 10.0
    dot_count    = name.count('.') / 10.0
    hyphen_count = name.count('-') / 5.0

    # ── Label (TIP ground truth, from exposure_score) ─────────
    if score == 10:
        sev = "Critical"
    elif score in [6, 7]:
        sev = "High"
    elif score <= 3 or verdict in ["Benign", "Unknown"]:
        sev = "Low"
    else:
        sev = "Medium"

    records.append({
        "indicator_type": ioc_type,
        "threat_category": threat_cat,
        "active_days":     active_days,
        "first_seen":      first_seen,
        "hour_created":    hour_created,
        "geo_risk":        geo_risk,
        "asn_reputation":  asn_rep,
        "name_len":        name_len,
        "dot_count":       dot_count,
        "hyphen_count":    hyphen_count,
        "severity":        sev
    })

df = pd.DataFrame(records)
print(f"Extracted {len(df)} records.")
print(df['severity'].value_counts())

In [ ]:
## 2. Sample Target Distribution (N = 3,000)

target_dist = {
    'Low': 1350,
    'Medium': 900,
    'High': 540,
    'Critical': 210
}

sampled_dfs = []
for label, count in target_dist.items():
    pool = df[df['severity'] == label]
    sampled = pool.sample(count, replace=(len(pool) < count), random_state=42)
    sampled_dfs.append(sampled)

study_df = pd.concat(sampled_dfs).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Final sampled distribution:")
print(study_df['severity'].value_counts())

In [ ]:
## 3. One-Hot Encoding and Split

X_cat = pd.get_dummies(study_df[['indicator_type', 'threat_category']], dtype=float)
X_num = study_df[['active_days', 'first_seen', 'hour_created',
                   'geo_risk', 'asn_reputation', 'name_len',
                   'dot_count', 'hyphen_count']]
X = pd.concat([X_cat, X_num], axis=1)

expected_features = [
    'indicator_type_Domain', 'indicator_type_Hash', 'indicator_type_IP',
    'threat_category_C2', 'threat_category_Exploit', 'threat_category_Malware',
    'threat_category_Phishing', 'threat_category_Other',
    'active_days', 'first_seen', 'hour_created',
    'geo_risk', 'asn_reputation', 'name_len', 'dot_count', 'hyphen_count'
]

X = X.reindex(columns=expected_features, fill_value=0.0)
y = study_df['severity']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape  : {X_val.shape}")
print(f"Classes      : {list(le.classes_)}")

In [ ]:
## 4. Random Forest Training & Evaluation

clf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=12,
    min_samples_leaf=5,
    random_state=42
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=le.classes_))

In [ ]:
## 5. Telemetry Plots

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            annot_kws={'size': 12, 'weight': 'bold'})
plt.ylabel('Actual Class', fontweight='bold')
plt.xlabel('Predicted Class', fontweight='bold')
plt.title('Fig 2. Random Forest Confusion Matrix\n(Automated Phase — Leakage-Free Features)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Data', 'confusion_matrix.png'), dpi=150)
plt.show()

# Plot 2: Feature Importance
importances = pd.Series(clf.feature_importances_, index=expected_features).sort_values()
plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='steelblue')
plt.xlabel('Mean Decrease in Impurity', fontweight='bold')
plt.title('Random Forest Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Data', 'feature_importance.png'), dpi=150)
plt.show()

# Save model & label encoder
with open(MODEL_OUT_PATH, 'wb') as f:
    pickle.dump(clf, f)
with open(ENCODER_OUT_PATH, 'wb') as f:
    pickle.dump(le, f)

print("Model saved to Data/ioc_rf_model.pkl")
print("Label encoder saved to Data/label_encoder.pkl")